---
# Part II – Tweets Emotion Classification using Word Embeddings

In [ ]:
import pandas as pd
import numpy as np
import gensim.downloader as gensim_api
from gensim.models import Word2Vec
from gensim.utils import simple_preprocess
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical

In [4]:
twdataset = pd.read_csv('twitter_emotion.csv')
print(twdataset.shape)
print(twdataset.head())

(416809, 3)
   Unnamed: 0                                               text  label
0           0      i just feel really helpless and heavy hearted      4
1           1  ive enjoyed being able to slouch about relax a...      0
2           2  i gave up my internship with the dmrg and am f...      4
3           3                         i dont know i feel so lost      0
4           4  i am a kindergarten teacher and i am thoroughl...      4


In [5]:
TWTEXTCOL  = twdataset.columns[0] 
TWLABELCOL = twdataset.columns[-1] 
print(f'Text column : {TWTEXTCOL}')
print(f'Label column: {TWLABELCOL}')
print('\nEmotion distribution:')
print(twdataset[TWLABELCOL].value_counts())

Text column : Unnamed: 0
Label column: label

Emotion distribution:
label
1    141067
0    121187
3     57317
4     47712
2     34554
5     14972
Name: count, dtype: int64


## 1. Tweets Pre-processing – Keras Tokenizer

In [6]:
MAX_WORDS  = 100_000
MAX_SEQ_LEN = 50         
tokenizer = Tokenizer(num_words=MAX_WORDS, oov_token='<OOV>')
tokenizer.fit_on_texts(twdataset[TWTEXTCOL].astype(str))
sequences = tokenizer.texts_to_sequences(twdataset[TWTEXTCOL].astype(str))
word_index = tokenizer.word_index
print(f'Vocab size        : {len(word_index)}')
print(f'First 10 entries  : {list(word_index.items())[:10]}')
padded_sequences = pad_sequences(
    sequences,
    maxlen=MAX_SEQ_LEN,
    padding='post',
    truncating='post'
)
print(f'Padded sequences shape: {padded_sequences.shape}')

Vocab size        : 416810
First 10 entries  : [('<OOV>', 1), ('0', 2), ('1', 3), ('2', 4), ('3', 5), ('4', 6), ('5', 7), ('6', 8), ('7', 9), ('8', 10)]
Padded sequences shape: (416809, 50)


## 2a. Pre-trained Embeddings – GloVe Twitter 50d

In [7]:
glove_model = gensim_api.load('glove-twitter-50')
EMBEDDING_DIM_GLOVE = 50
print(f'Vocabulary size in GloVe model: {len(glove_model.key_to_index)}')

Vocabulary size in GloVe model: 1193514


In [8]:
def buildEmbeddingMatrix(wvModel, word_index, max_words, embedding_dimension):
    matrix = np.zeros((max_words + 1, embedding_dimension))
    for word, idx in word_index.items():
        if idx > max_words:
            continue
        try:
            matrix[idx] = wvModel[word]
        except KeyError:
            pass   
    return matrix
glove_embedding_matrix = buildEmbeddingMatrix(
    glove_model, word_index, MAX_WORDS, EMBEDDING_DIM_GLOVE
)
print(f'GloVe embedding matrix shape: {glove_embedding_matrix.shape}')

GloVe embedding matrix shape: (100001, 50)


## 2b. Your trained word2vec model word embeddings:

In [23]:
print("Step 2b – Training custom Word2Vec model …")

indx_to_word = {idx: word for word, idx in word_index.items()}
tokenized_tweets = [
    [indx_to_word[idx] for idx in seq if idx in indx_to_word]
    for seq in sequences
]

print(f"Total tokenized tweets : {len(tokenized_tweets)}")
print(f"First tokenized tweet  : {tokenized_tweets[0]}")

w2v_model = Word2Vec(
    sentences=tokenized_tweets,
    vector_size=100,
    window=5,
    min_count=1,
    epochs=10,
    sg=1
)

w2v_model.save("word2vec_twitter.model")
print(f"Word2Vec size : {len(w2v_model.wv.key_to_index)}")

print("Building Word2Vec embedding matrix")
w2v_embedding_matrix = buildEmbeddingMatrix(
    w2v_model.wv, word_index, MAX_WORDS, 100
)
print(f"Word2Vec embedding matrix shape: {w2v_embedding_matrix.shape}")

Step 2b – Training custom Word2Vec model …
Total tokenized tweets : 416809
First tokenized tweet  : ['0']
Word2Vec size : 99999
Building Word2Vec embedding matrix
Word2Vec embedding matrix shape: (100001, 100)


## 3. Data preparation and splitting:

3a. Apply one-hot encoding for the integer labels using keras.

In [24]:
from tensorflow.keras.utils import to_categorical

labels = twdataset[TWLABELCOL].values
one_hot_labels = to_categorical(labels)

print("Original labels shape:", labels.shape)
print("One-hot labels shape:", one_hot_labels.shape)

Original labels shape: (416809,)
One-hot labels shape: (416809, 6)


3b. Splitting


In [25]:
from sklearn.model_selection import train_test_split

# split 80% & 20%

X_train_80, X_test_80, y_train_80, y_test_80 = train_test_split(
    padded_sequences,       # input features
    one_hot_labels,         # labels
    test_size=0.2,          # 20% test
    random_state=42,        
    shuffle=True
)

print("\n80/20 Split:")
print("X_train shape:", X_train_80.shape)
print("X_test shape :", X_test_80.shape)


# split 70% & 30%

X_train_70, X_test_70, y_train_70, y_test_70 = train_test_split(
    padded_sequences,
    one_hot_labels,
    test_size=0.3,          
    random_state=42,
    shuffle=True
)

print("\n70/30 Split:")
print("X_train shape:", X_train_70.shape)
print("X_test shape :", X_test_70.shape)


80/20 Split:
X_train shape: (333447, 50)
X_test shape : (83362, 50)

70/30 Split:
X_train shape: (291766, 50)
X_test shape : (125043, 50)
